# VoxConverse Diarization Benchmark Notebook

This notebook consolidates the complete BKM workflow for PyTorch and OpenVINO diarization benchmarking on VoxConverse.

Backends covered:
- OpenVINO CPU
- OpenVINO GPU
- PyTorch CPU
- PyTorch XPU


## Workflow

- Prepare repository
- Install Miniforge (if needed)
- Create conda environments
- Configure Hugging Face access
- Download VoxConverse
- Export OpenVINO IR
- Run smoke tests
- Run 80-100 second benchmarks
- Run full DER scoring
- Troubleshoot common pitfalls

## Notebook Setup

Detects this notebook's folder so all commands below work regardless of username or clone location.

In [ ]:
import os

# Resolve this notebook's own folder (works in VS Code Jupyter; falls back to cwd otherwise)
NOTEBOOK_DIR = os.path.dirname(os.path.abspath(globals().get("__vsc_ipynb_file__", os.getcwd())))
os.environ["NOTEBOOK_DIR"] = NOTEBOOK_DIR
print(f"NOTEBOOK_DIR = {NOTEBOOK_DIR}")

## Miniforge Installation

Skip this if conda is already available at ~/miniforge3.

In [ ]:
%%bash
set -euo pipefail
if [ ! -d "$HOME/miniforge3" ]; then
  curl -L -O "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"
  bash Miniforge3-Linux-x86_64.sh -b -p "$HOME/miniforge3"
  rm -f Miniforge3-Linux-x86_64.sh
fi
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda --version

## Environment Creation

In [ ]:
%%bash
set -euo pipefail
source ~/miniforge3/etc/profile.d/conda.sh

create_if_missing() {
  env_name="$1"
  yaml_file="$2"
  if conda env list | awk '{print $1}' | grep -qx "$env_name"; then
    echo "Environment '$env_name' already exists. Skipping creation."
  else
    echo "Creating environment '$env_name' from $yaml_file ..."
    conda env create -f "$yaml_file"
  fi
}

create_if_missing diar_cpu diar_cpu.yaml
create_if_missing diar_ov diar_ov.yaml
create_if_missing diar_xpu diar_xpu.yaml

### Verify environments

Lists the conda environments and filters for `diar_cpu`, `diar_ov`, and `diar_xpu` to confirm all three were created.

In [ ]:
%%bash
set -euo pipefail
source ~/miniforge3/etc/profile.d/conda.sh
conda env list | grep -E "diar_cpu|diar_ov|diar_xpu"

## Select the notebook kernel by environment

Use the VS Code notebook kernel selector and choose the matching environment before running cells:
- `diar_cpu` for PyTorch CPU
- `diar_xpu` for XPU
- `diar_ov` for OpenVINO CPU/GPU

This notebook is grouped into three sections below so you can run the relevant commands for each backend without mixing environments.

## Hugging Face Access

Create a Read token at https://huggingface.co/settings/tokens, then accept model terms:
- https://hf.co/pyannote/speaker-diarization-community-1
- https://hf.co/pyannote/segmentation-3.0

In [ ]:
%%bash
set -euo pipefail
source ~/miniforge3/etc/profile.d/conda.sh
conda activate diar_ov
huggingface-cli login
conda deactivate

## VoxConverse Dataset Download

In [ ]:
%%bash
set -euo pipefail
cd "$NOTEBOOK_DIR"
bash download_voxconverse.sh --dest ./voxconverse

### Verify dataset layout

Confirms the expected VoxConverse `dev`, `test`, and wav folders exist after download.

In [ ]:
%%bash
set -euo pipefail
cd "$NOTEBOOK_DIR"
ls -d ./voxconverse/dev ./voxconverse/test ./voxconverse/voxconverse_dev_wav ./voxconverse/voxconverse_test_wav

## OpenVINO CPU/GPU (diar_ov)

Important: switch the notebook kernel to `diar_ov` before running the cells in this section.

Use the `diar_ov` kernel for all OpenVINO CPU/GPU cells below.

### Export OpenVINO IR

Exports the diarization neural blocks to OpenVINO IR (`.xml` / `.bin`) into `./ov_models`. Run this once before the OpenVINO smoke tests.

In [ ]:
%%bash
set -euo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
conda activate diar_ov
python export_pyann.py --output-dir ./ov_models
conda deactivate

### Smoke test on CPU

Runs OpenVINO diarization on CPU for a single file to confirm the IR loads and runs.

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_ov
python run_diarization_ov.py voxconverse/voxconverse_test_wav/voxconverse_test_wav/nqcpi.wav --device CPU
conda deactivate

### Smoke test on GPU

Same single-file run on the Intel iGPU (`--device GPU`) to verify the OpenVINO GPU path. Inference runs in FP16 (OpenVINO's default precision on Intel GPUs, with FP16-compressed IR weights).

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_ov
python run_diarization_ov.py voxconverse/voxconverse_test_wav/voxconverse_test_wav/nqcpi.wav --device GPU
conda deactivate

### 80-100s file benchmark (CPU and GPU)

Times the short 80-100 second files for both OpenVINO CPU and iGPU backends.

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_ov
bash run_file_benchmark.sh --backends "ov-cpu" --vox-root ./voxconverse
bash run_file_benchmark.sh --backends "ov-gpu" --vox-root ./voxconverse
conda deactivate

### Full DER scoring (CPU and GPU)

Runs the full VoxConverse test split for OpenVINO CPU and iGPU, writing DER results to `voxcon_ov_cpu` and `voxcon_ov_igpu`.

Note: the originally committed DER was 11.2%, but this run is currently getting 8.3%.

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_ov
python score_der.py --dataset voxconverse --subset test --backend ov-cpu --vox-root ./voxconverse --ir-dir ./ov_models | tee voxcon_ov_cpu
python score_der.py --dataset voxconverse --subset test --backend ov-gpu --vox-root ./voxconverse --ir-dir ./ov_models | tee voxcon_ov_igpu
conda deactivate

## PyTorch (diar_cpu)

Important: switch the notebook kernel to `diar_cpu` before running the cells in this section.

Use the `diar_cpu` kernel for all PyTorch CPU cells below.

### Smoke test (single file)

Runs diarization on one VoxConverse test file to confirm the PyTorch CPU setup works end to end before larger runs.

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_cpu
python run_diarization.py voxconverse/voxconverse_test_wav/voxconverse_test_wav/nqcpi.wav
conda deactivate

### 80-100s file benchmark

Times diarization on the short 80-100 second files using the helper script, saved automatically to a benchmark log.

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_cpu
bash run_file_benchmark.sh --backends "cpu" --vox-root ./voxconverse
conda deactivate

### Full DER scoring

Runs the full VoxConverse test split and computes Diarization Error Rate, writing results to `voxcon_cpu`.

Note: the originally committed DER was 11.2%, but this run is currently getting 8.3%.

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_cpu
python score_der.py --dataset voxconverse --subset test --backend cpu --vox-root ./voxconverse | tee voxcon_cpu
conda deactivate

## XPU (diar_xpu)

Important: switch the notebook kernel to `diar_xpu` before running the cells in this section.

Use the `diar_xpu` kernel for all XPU cells below.

### Smoke test (single file)

Runs diarization on one file with `--device xpu` to verify the Intel XPU (GPU) path is working before larger runs.

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_xpu
python run_diarization.py voxconverse/voxconverse_test_wav/voxconverse_test_wav/nqcpi.wav --device xpu
conda deactivate

### 80-100s file benchmark

Times the short 80-100 second files on XPU; results are saved to `file_benchmark_xpu.log`.

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_xpu
bash run_file_benchmark.sh --backends "xpu" --vox-root ./voxconverse
conda deactivate

### Full DER scoring

Runs the full VoxConverse test split on XPU and writes DER results to `voxcon_xpu`.

Note: the originally committed DER was 11.2%, but this run is currently getting 8.3%.

In [ ]:
%%bash
set -eo pipefail
cd "$NOTEBOOK_DIR"
source ~/miniforge3/etc/profile.d/conda.sh
set +u
conda activate diar_xpu
python score_der.py --dataset voxconverse --subset test --backend xpu --vox-root ./voxconverse | tee voxcon_xpu
conda deactivate

## Summary of all DER results

Extracts the total DER line from every backend log (`voxcon_cpu`, `voxcon_xpu`, `voxcon_ov_cpu`, `voxcon_ov_igpu`) for a side-by-side comparison.

In [ ]:
%%bash
set -euo pipefail
cd "$NOTEBOOK_DIR"
found=0
for log in voxcon_cpu voxcon_xpu voxcon_ov_cpu voxcon_ov_igpu; do
  if [ -f "$log" ]; then
    found=1
    grep "^# TOTAL" "$log" 2>/dev/null || echo "$log: no TOTAL line found"
  else
    echo "$log: not found, skipping"
  fi
done
if [ "$found" -eq 0 ]; then
  echo "No DER log files found yet. Run the scoring cells first."
fi